# Climate Data Downloader

Downloads air temperature and soil temperature from four paleoclimate datasets:

| Dataset | Source | Air temp | Soil temp |
|---|---|---|---|
| **TraCE-21k** | NCAR | `TSA` | `TSOI` |
| **CHELSA-TraCE21k-centennial** | EnviDat | `tasmax`, `tasmin` | - |
| **Beyer2020** | figshare | (bundled) | - |
| **PalMod2** | WDCC / DKRZ | `tas` | `tsl` |

## Setup

PalMod2 requires a WDCC account. Create an account and write your credentials to `PalMod2_credentials.txt` in the notebook directory:

- Line 1: username
- Line 2: password

In [1]:
import os
from pathlib import Path
import requests
from urllib.request import build_opener
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pexpect
import getpass
import sys

## Download TraCE-21k

Reads URLs from `data/TraCE-21k/download_paths.txt` and downloads:

- `TSA` — near-surface air temperature
- `TSOI` — soil temperature

In [2]:
txt_path = Path.cwd() / 'data' / 'TraCE-21k' / 'download_paths.txt'
output_path = Path.cwd() / 'data' / 'TraCE-21k' / 'data'
output_path.mkdir(parents=True, exist_ok=True)

opener = build_opener()

with open(txt_path, "r") as f:
    full_filelist = [line.strip() for line in f if line.strip()]

keep = {"TSA", "TSOI"}
filelist = [url for url in full_filelist if any(f".{var}." in url for var in keep)]

for file in tqdm(filelist):
    ofile = os.path.basename(file)
    dest = output_path / ofile

    if dest.exists():
        #tqdm.write(f"Skipping {ofile} (already exists)")
        continue

    #tqdm.write(f"Downloading {ofile} ...")
    infile = opener.open(file)
    with open(dest, "wb") as outfile:
        outfile.write(infile.read())
    #tqdm.write(f"Done: {ofile}")

  0%|          | 0/2 [00:00<?, ?it/s]

## Download CHELSA-TraCE21k-centennial

Reads URLs from `data/CHELSA-TraCE21k-centennial/envidatS3paths.txt` and downloads:

- `tasmax` — daily maximum near-surface air temperature
- `tasmin` — daily minimum near-surface air temperature

In [ ]:
txt_path = Path.cwd() / 'data' / 'CHELSA-TraCE21k-centennial' / 'envidatS3paths.txt'
output_path = Path.cwd() / 'data' / 'CHELSA-TraCE21k-centennial' / 'data'
output_path.mkdir(parents=True, exist_ok=True)

with open(txt_path) as f:
    all_urls = [line.strip() for line in f if line.strip()]

CHUNK = 1024 * 1024  # 1 MiB

def download(url):
    dest = output_path / os.path.basename(url)
    if dest.exists():
        return
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(tmp, "wb") as out:
            for chunk in r.iter_content(chunk_size=CHUNK):
                if chunk:
                    out.write(chunk)
    tmp.rename(dest)

variables = {
    "tasmax", 
    "tasmin"
}
urls = [u for u in all_urls if any(f"_{v}_" in u for v in variables)]

with ThreadPoolExecutor(max_workers=32) as ex:
    futures = [ex.submit(download, u) for u in urls]
    for fut in tqdm(as_completed(futures), total=len(urls)):
        try:
            fut.result()
        except Exception as e:
            tqdm.write(f"Failed: {e}")

  0%|          | 0/5305 [00:00<?, ?it/s]

## Download Beyer2020

Queries the figshare API for all files in version 4 of the dataset (article ID `12293345`) and downloads them.

- All variables (single bundled NetCDF)

In [ ]:
ARTICLE_ID = 12293345
#VERSION = 3  # Beyer2020
VERSION = 4  # Beyer2021 (2021 update, adds monthly NPP)
API_URL = f"https://api.figshare.com/v2/articles/{ARTICLE_ID}/versions/{VERSION}/files"

# download LateQuaternary_Environment data
output_path = Path.cwd() / 'data' / 'Beyer2020' / 'data'
output_path.mkdir(parents=True, exist_ok=True)

response = requests.get(API_URL, timeout=60)
response.raise_for_status()
filelist = response.json()  # list of dicts with 'name' and 'download_url'

for file in tqdm(filelist):
    ofile = file['name']
    dest = output_path / ofile
    if dest.exists():
        continue
    with requests.get(file['download_url'], stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(dest, 'wb') as outfile:
            for chunk in r.iter_content(chunk_size=8192):
                outfile.write(chunk)

## Download PalMod2

Uses the `jblob` command-line tool (v4.1) to download from WDCC. Login is automated via `pexpect`.

- `tas` — near-surface air temperature (monthly)
- `tsl` — soil temperature by layer (monthly)

**Credentials:** create `PalMod2_credentials.txt` with WDCC username on line 1 and password on line 2, or hardcode them in the cell below.

In [ ]:
with open("PalMod2_credentials.txt") as file:
    lines = file.read().splitlines()
    username = lines[0]
    password = lines[1]

jblob_path = Path.cwd() / 'data' / 'PalMod2' / 'jblob-4.1' / 'jblob.sh'

output_path = Path.cwd() / 'data' / 'PalMod2' / 'data'
output_path.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "tas": "PMMXMCRTDGr111Amtasgn30201",  # near-surface air temperature, monthly
    "tsl": "PMMXMCRTDGr111Lmtslgn30201",  # soil temperature by layer, monthly
}

# login to wdcc
child = pexpect.spawn(f"bash {jblob_path} --login", encoding="utf-8", timeout=60)
child.logfile = None  # set to sys.stdout to debug

child.expect("Account type.*:")
child.sendline("WDCC")

child.expect("Username.*:")
child.sendline(username)

child.expect("Password.*:")
child.sendline(password)

child.expect(["Login successful", pexpect.EOF])
print("Login:", child.before + child.after)
child.close()

# download
for name, dataset_id in DATASETS.items():
    # skip if already downloaded
    if list(output_path.glob(f"*{dataset_id}*")):
        print(f"Skipping {name} (already exists)")
        continue

    print(f"\nDownloading {name} ({dataset_id}) ...")
    child = pexpect.spawn(
        f"bash {jblob_path} --dataset {dataset_id} --dir {output_path}",
        encoding="utf-8", timeout=None
    )
    child.logfile = None
    child.expect(pexpect.EOF)
    print(f"Done: {name}")

print("\nAll downloads complete.")